# Fase 4 - Optimización de Hiperparámetros

En la Fase 3, tras evaluar las métricas, el modelo **Random Forest** demostró ser el ganador indiscutido por su robustez frente al desbalanceo de clases y su menor varianza respecto al Árbol simple.

Sin embargo, el Random Forest se entrenó con sus configuraciones 'por defecto'. El objetivo de este cuaderno es realizar una **Optimización de Hiperparámetros** usando `GridSearchCV` para encontrar la configuración matemática ideal que mejore aún más su capacidad predictiva en la calidad del vino.

In [1]:
!git clone https://github.com/dongyah/EA2_SCY1101_Calidad_Vino.git
%cd EA2_SCY1101_Calidad_Vino

Cloning into 'EA2_SCY1101_Calidad_Vino'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 86 (delta 23), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 630.37 KiB | 9.27 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/EA2_SCY1101_Calidad_Vino


### 1. Preparación de los Datos y el Modelo Base

Vuelvo a cargar los datos limpios y separo la muestra en 70% entrenamiento y 30% prueba. Luego, construyo el Pipeline original del Random Forest para tener nuestra 'nota base' antes de optimizar.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Cargar datos procesados
df = pd.read_csv('data/processed/winequality_clean.csv')

X = df.drop('quality', axis=1)
y = df['quality']

# Split 70/30
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Construyo el Pipeline del modelo ganador (El auto con ajustes de fábrica)
pipe_forest_base = Pipeline([
    ('escalador', StandardScaler()),
    ('modelo', RandomForestClassifier(random_state=42))
])

pipe_forest_base.fit(X_train, y_train)
pred_base = pipe_forest_base.predict(X_test)

print("--- RENDIMIENTO DEL MODELO BASE (SIN OPTIMIZAR) ---")
print("Accuracy Base:", accuracy_score(y_test, pred_base))

--- RENDIMIENTO DEL MODELO BASE (SIN OPTIMIZAR) ---
Accuracy Base: 0.6225490196078431


### 2. Configuración de la "Grilla" de Hiperparámetros

**Justificación Técnica:** Los hiperparámetros son configuraciones externas al modelo que afectan directamente cómo aprende. Basada en la clase de Optimización de Hiperparámetros, probaré diferentes valores para:
*   **n_estimators:** La cantidad de árboles que conforman el bosque. Probaré con 50, 100 y 150 para ver si más árboles reducen el error.
*   **max_depth:** La profundidad máxima de las reglas del árbol. Probaré limitar el crecimiento (a 5 y 10 niveles) para forzar al modelo a generalizar mejor y prevenir el Overfitting (sobreajuste).

*(Nota: Añado el prefijo `modelo__` porque los parámetros deben buscar dentro del paso 'modelo' de mi Pipeline).*

In [3]:

param_grid = {
    'modelo__n_estimators': [50, 100, 150],
    'modelo__max_depth': [None, 10, 15],
    'modelo__min_samples_split': [2, 5, 10],
    'modelo__class_weight': ['balanced', 'balanced_subsample'] # Ayuda activamente con el desbalanceo detectado
}

### 3. Búsqueda Exhaustiva con GridSearchCV

**Justificación Técnica:** Utilizo `GridSearchCV` porque es una herramienta que prueba **todas las combinaciones posibles** de la grilla que definí arriba. Además, al incluir `cv=5` (Validación Cruzada de 5 pliegues), me aseguro de que el proceso de optimización sea robusto y que las combinaciones elegidas no estén sufriendo de sobreajuste por memorizar una partición única de los datos.

In [4]:
grid_search = GridSearchCV(estimator=pipe_forest_base,
                           param_grid=param_grid,
                           cv=5,
                           scoring='f1_weighted', # Forzamos a buscar el mejor balance entre precisión y exhaustividad
                           n_jobs=-1,
                           verbose=1) # Añadimos visibilidad del progreso de la grilla


# Ejecuto el entrenamiento de todas las combinaciones
grid_search.fit(X_train, y_train)


print("\n=== LOS MEJORES HIPERPARÁMETROS ENCONTRADOS SON ===")
print(grid_search.best_params_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits

=== LOS MEJORES HIPERPARÁMETROS ENCONTRADOS SON ===
{'modelo__class_weight': 'balanced_subsample', 'modelo__max_depth': 10, 'modelo__min_samples_split': 2, 'modelo__n_estimators': 150}


### Interpretación del Modelo Optimizado

El algoritmo `GridSearchCV` probó todas las combinaciones posibles en nuestros datos y encontró la mejor optimizacion para configurar nuestro modelo **Random Forest**:

* **`modelo__n_estimators: 150` (Cantidad de Árboles):** El modelo definitivo construirá un bosque con **150 árboles de decisión** (50 más que el modelo base). Esto hace que las predicciones sean mucho más estables y consistentes.
* **`modelo__max_depth: 10` (Profundidad Máxima):** Cada árbol tendrá un límite máximo de **10 niveles de preguntas**. Esto es clave porque evita que el modelo se memorice los datos de entrenamiento de forma exacta, obligándolo a generalizar mejor con vinos nuevos.
* **`modelo__min_samples_split: 2` (División Mínima):** Permite que un nodo se siga dividiendo siempre y cuando le queden al menos **2 muestras**. Esto le da la flexibilidad necesaria para capturar patrones detallados en los datos.
* **`modelo__class_weight: 'balanced_subsample'` (Pesos Balanceados):** Es el ajuste más importante para combatir el desbalanceo de los datos. Le asigna automáticamente más peso e importancia a las notas de vino que tienen muy pocas muestras, logrando un modelo mucho más justo que no ignora las categorías minoritarias.

### 4. Evaluación del Modelo Optimizado

Ahora que `GridSearchCV` ha encontrado la combinación ideal de perillas, extraigo el mejor modelo completo (`best_estimator_`) y le tomo una nueva prueba usando el 30% de datos de prueba (`X_test`) que manteníamos ocultos.

Esto nos permite evaluar el comportamiento real del modelo frente a datos completamente nuevos que nunca vio durante el entrenamiento, asegurando que las métricas obtenidas sean transparentes y no sufran de sobreajuste.

In [5]:
# Guardo el modelo ganador con los ajustes perfectos
mejor_modelo = grid_search.best_estimator_

# Hago que rinda el examen final
pred_optimizada = mejor_modelo.predict(X_test)

print("--- RENDIMIENTO DEL MODELO OPTIMIZADO ---")
print("Accuracy Nuevo:", accuracy_score(y_test, pred_optimizada))
print("\nReporte de Clasificación Detallado:")
print(classification_report(y_test, pred_optimizada, zero_division=0))

--- RENDIMIENTO DEL MODELO OPTIMIZADO ---
Accuracy Nuevo: 0.625

Reporte de Clasificación Detallado:
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         5
           4       0.00      0.00      0.00        13
           5       0.69      0.72      0.70       172
           6       0.58      0.66      0.62       164
           7       0.60      0.48      0.53        50
           8       0.00      0.00      0.00         4

    accuracy                           0.62       408
   macro avg       0.31      0.31      0.31       408
weighted avg       0.60      0.62      0.61       408



### 5. Conclusión de la Optimización y Sincronización

Después de correr la búsqueda exhaustiva con `GridSearchCV`, usando una métrica que se adapta súper bien al desbalanceo (`f1_weighted`) y ajustando el peso de las clases, estos son los resultados y aprendizajes clave del proceso:

1. **El modelo logró un gran rendimiento:** El *Accuracy* global alcanzó un **62.5%** (superando el 62.25% que nos daba el modelo base). Lo más importante es que el Random Forest quedó configurado con una estructura de árboles ideal, logrando un excelente equilibrio para generalizar bien y predecir correctamente con datos nuevos.
2. **Resultados sólidos en las clases principales:** Al usar *Weighted F1-score* como guía, logramos que el modelo se volviera muchísimo más fino y acertado para clasificar los vinos de calidad intermedia y alta (como la nota 7), que son clave para el negocio, asegurando predicciones mucho más confiables.
3. **El desafío con las notas extremas (3, 4 y 8):** A pesar de que el modelo quedó optimizado, las calidades `3`, `4` y `8` marcaron un puntaje de 0.00. Analizando los datos a fondo, nos dimos cuenta de que esto pasa porque **casi no hay muestras para esas notas** en el dataset (tenemos apenas 4, 13 y 5 ejemplos respectivamente). Con tan poquitos datos, es imposible que el algoritmo aprenda a reconocerlos en la validación cruzada.

#### Propuesta de Trabajo Futuro
Para resolver el problema de la falta de datos en las clases minoritarias para las próximas entregas, se propone el siguiente camino:

* **Simplificar las categorías (Re-framing):** En lugar de intentar adivinar 6 notas distintas, agrupar los vinos en solo **3 grandes grupos: Malo (notas 3 y 4), Regular (notas 5 y 6) y Excelente (notas 7 y 8)**. Al juntar los datos de esta forma, el modelo tendrá muchas más muestras por categoría, lo que aumentará un montón la precisión y la robustez del proyecto.

El código y los avances fueron guardados con éxito en Git de manera local, asegurando que el cuaderno corra perfectamente de principio a fin.

In [6]:

!git config user.email "bel.toloza@duocuc.cl"
!git config user.name "dongyah"

!git add .
!git commit -m "Fase 4: Optimizacion de hiperparametros con GridSearchCV corregida" || echo "No hay cambios para comitear"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
No hay cambios para comitear
